In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!pip install transformers==4.38.2 accelerate==0.27.2 peft==0.9.0 timm torchvision -q

In [ ]:
import os
os.kill(os.getpid(), 9)

In [ ]:
lora_path = "/content/drive/MyDrive/OCR/OCRMOD"

In [ ]:
import torch
from PIL import Image
from transformers import AutoTokenizer, AutoModel
from peft import PeftModel

# ==============================
# PATHS
# ==============================
base_model_name = "openbmb/MiniCPM-V-2"
lora_path = "/content/drive/MyDrive/OCR/OCRMOD"

# ==============================
# LOAD TOKENIZER
# ==============================
tokenizer = AutoTokenizer.from_pretrained(
    base_model_name,
    trust_remote_code=True
)

# ==============================
# LOAD MODEL (خفيف)
# ==============================
model = AutoModel.from_pretrained(
    "openbmb/MiniCPM-V-2",
    trust_remote_code=True,
    torch_dtype=torch.float16,
    device_map="auto",
    low_cpu_mem_usage=True
)

# ==============================
# LOAD LORA
# ==============================
model = PeftModel.from_pretrained(model, lora_path)

model = model.half()
model.eval()

print("✅ Model Ready")

# ==============================
# PREDICT
# ==============================
def predict(image_path):
    image = Image.open(image_path).convert("RGB")

    msgs = [{
        "role": "user",
        "content": "Extract ONLY the drug name from this image."
    }]

    model.eval()

    with torch.no_grad():
        res = model.chat(
            image=image,
            msgs=msgs,
            context=None,
            tokenizer=tokenizer
        )

    return res



In [ ]:
import torch
import gc

gc.collect()
torch.cuda.empty_cache()

In [ ]:
# ==============================
# TEST
# ==============================
from PIL import Image
image_path = "/content/drive/MyDrive/OCR/57376930image.webp"
image = Image.open(image_path).convert("RGB")
image = image.resize((224, 224))  # أو 256
print(predict(image_path))

In [ ]:
# ==============================
# TEST
# ==============================
from PIL import Image
image_path = "/content/drive/MyDrive/OCR/57376930image.webp"
image = Image.open(image_path).convert("RGB")
image = image.resize((224, 224))  # أو 256
print(predict(image_path))

In [ ]:
# ==============================
# TEST
# ==============================
from PIL import Image
image_path = "/content/drive/MyDrive/OCR/fd22bc4b-7fe2-46d0-a0a5-633c8c332683-1000x1000-CyuIdnZ4NsyfF8JWSIo1K2vO0NUPPCL8hsUJQlpB.webp"
image = Image.open(image_path).convert("RGB")
image = image.resize((224, 224))  # أو 256
print(predict(image_path))

In [ ]:
  # ==============================
# TEST
# ==============================
from PIL import Image
image_path = "/content/drive/MyDrive/OCR/21.jpeg"
image = Image.open(image_path).convert("RGB")
image = image.resize((224, 224))  # أو 256
print(predict(image_path))

In [ ]:
import matplotlib.pyplot as plt
from PIL import Image

def show_prediction(image_path):
    # افتح الصورة
    image = Image.open(image_path).convert("RGB")

    # اعمل prediction
    result = predict(image_path)

    # النتيجة بتاعتك شكلها tuple → ناخد أول عنصر
    pred_text = result[0]

    # عرض الصورة
    plt.imshow(image)
    plt.axis("off")

    # حط التايتل = اسم الدواء
    plt.title(f"Prediction: {pred_text}", fontsize=14)

    plt.show()


# test
image_path = "/content/drive/MyDrive/OCR/21.jpeg"
show_prediction(image_path)

In [ ]:
# test
image_path = "/content/drive/MyDrive/OCR/57376930image.webp"
show_prediction(image_path)

In [ ]:
# test
image_path = "/content/drive/MyDrive/OCR/fd22bc4b-7fe2-46d0-a0a5-633c8c332683-1000x1000-CyuIdnZ4NsyfF8JWSIo1K2vO0NUPPCL8hsUJQlpB.webp"
show_prediction(image_path)

In [ ]:
import nest_asyncio
import uvicorn
import io
import threading
from fastapi import FastAPI, File, UploadFile, HTTPException
from fastapi.responses import JSONResponse
from PIL import Image

nest_asyncio.apply()

app = FastAPI(title="Drug OCR API", version="1.0.0")


@app.get("/health")
def health_check():
    return {"status": "ok", "model": "MiniCPM-V-2 + LoRA"}


@app.post("/predict")
async def predict_drug(file: UploadFile = File(...)):
    try:
        contents = await file.read()
        image = Image.open(io.BytesIO(contents)).convert("RGB")

        temp_path = "/tmp/uploaded_drug_image.jpg"
        image.save(temp_path)

        result = predict(temp_path)
        drug_name = result[0] if isinstance(result, (list, tuple)) else result

        return JSONResponse(content={"drug_name": drug_name.strip()})

    except Exception as e:
        raise HTTPException(status_code=500, detail=f"Prediction failed: {str(e)}")


def run_fastapi_server():
    uvicorn.run(app, host="0.0.0.0", port=8000, log_level="info")

server_thread = threading.Thread(target=run_fastapi_server, daemon=True)
server_thread.start()

print("✅ FastAPI Server Running on port 8000")

In [ ]:
from pyngrok import ngrok

# أوقف أي tunnel قديم
ngrok.kill()

# Store your token in Colab Secrets (key icon in the sidebar) under the name NGROK_TOKEN
from google.colab import userdata
ngrok.set_auth_token(userdata.get("NGROK_TOKEN"))

public_url = ngrok.connect(8000)
print("🌐 Public URL:", public_url)

In [ ]:
import requests
from google.colab import files

uploaded = files.upload()
filename = list(uploaded.keys())[0]

# Use the public URL printed by the previous cell
url = f"{public_url.public_url}/predict"

with open(filename, "rb") as f:
    response = requests.post(url, files={"file": (filename, f)})

# مش .json() - عايزين نشوف الـ raw text
print("Status Code:", response.status_code)
print("Raw Response:", response.text)